# Kairos: фичи относительно автора и модель

Нужны результаты `embed.ipynb` в `MyDrive/kairos_work/`: `text_e5.npz` обязательно, `img/part_*.npz` если есть
(без них ноутбук работает только с текстом). Рантайм: CPU с High-RAM или любой GPU-рантайм (GPU не используется,
но у таких машин больше ядер и памяти). Пароль нужен, только если `light.tar` ещё не создан.

Идея: метка считается относительно автора, поэтому главное не «какой пост», а «чем пост отличается от прошлых постов этого автора
и как те прошлые посты сыграли». Все фичи истории читают только более ранние посты, как и сам `level`.

Пароль от архива берётся из секрета Colab `KAIROS_PW` (значок ключа слева, «Add new secret»,
включить «Notebook access»). Если секрета нет, то из переменной окружения `KAIROS_PW`, и только потом спрашивается.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, json, time, shutil, subprocess, getpass, warnings
import numpy as np, pandas as pd

ARCHIVES = '/content/drive/MyDrive/kairos_archives'
ARCH = f'{ARCHIVES}/lab1_core.7z.001'
WORK = '/content/drive/MyDrive/kairos_work'      # результаты, на Drive
DATA = '/content/kairos_data'                     # тексты и метки, локальный диск VM
MEDIA = '/content/kairos_media'                   # временная распаковка медиа
REPO = '/content/kairos-dataset'
os.makedirs(f'{WORK}/img', exist_ok=True)

_pw = None
def password():
    # Секрет Colab, потом переменная окружения, потом ручной ввод; спрашиваем один раз.
    global _pw
    if _pw is None:
        try:
            from google.colab import userdata
            _pw = userdata.get('KAIROS_PW')
        except Exception:
            _pw = os.environ.get('KAIROS_PW') or getpass.getpass('Пароль от архива: ')
    return _pw

def _z(args):
    return ['7z', *args, f'-p{password()}', '-y', '-mmt=on']

def z(*args):
    # 7z синхронно, возвращает stdout.
    r = subprocess.run(_z(args), capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-2000:] or r.stdout[-2000:])
    return r.stdout

def z_start(*args):
    # 7z в фоне: распаковка идёт, пока ноутбук занят другим.
    return subprocess.Popen(_z(args), stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)

def z_wait(p):
    _, err = p.communicate()
    if p.returncode != 0:
        raise RuntimeError(err[-2000:])

subprocess.run('which 7z >/dev/null || apt-get -qq install -y p7zip-full', shell=True, check=True)

In [ ]:
# Оглавление архива и лёгкая часть релиза (тексты, метки, manifest).
# Первый раз из архива, дальше из кеша на Drive: распаковка tar занимает секунды.
TOC, LIGHT = f'{WORK}/toc.parquet', f'{WORK}/light.tar'
if os.path.exists(TOC):
    toc = pd.read_parquet(TOC)
else:
    rows, cur = [], {}
    for line in z('l', '-ba', '-slt', ARCH).splitlines():
        if line.startswith('Path = '):
            cur = {'path': line[7:]}
            rows.append(cur)
        elif line.startswith('Size = ') and cur:
            cur['size'] = int(line[7:] or 0)
    toc = pd.DataFrame(rows)
    toc.to_parquet(TOC)
PREFIX = toc.loc[toc['path'].str.endswith('labels.csv'), 'path'].iloc[0][:-len('labels.csv')]

if not os.path.isdir(DATA):
    if os.path.exists(LIGHT):
        subprocess.run(['tar', 'xf', LIGHT, '-C', '/content'], check=True)
    else:
        wanted = [PREFIX + p for p in ('labels.csv', 'split.json', 'media/manifest.jsonl', 'posts', 'authors')]
        z('x', ARCH, f'-o{DATA}', *wanted)
        subprocess.run(['tar', 'cf', LIGHT, '-C', '/content', 'kairos_data'], check=True)
DATA_DIR = os.path.join(DATA, PREFIX) if PREFIX else DATA

if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/Danil482/kairos-dataset.git', REPO], check=True)
sys.path.insert(0, REPO)
from kairos.data import load_posts, load_labels, load_split, load_followers

posts = load_posts(DATA_DIR)          # только свои посты, вместе с неразмеченной историей
print(f'{len(posts):,} own posts, данные в {DATA_DIR}')

## 1. Фичи бейзлайна и счётчики из фидов

`commentsCount`, `repostsCount` и т.п. у самого поста это отклик на него, то есть почти таргет.
Их берём **только как историю**: среднее по прошлым постам автора.
Фиды разбираются параллельно в процессах (json-парсинг упирается в GIL).

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from kairos import features
from kairos.label import author_level

split = load_split(DATA_DIR)
LOW, HIGH = split['cutoffs_on_log_residual']['low'], split['cutoffs_on_log_residual']['high']
TRAIN_AUTHORS = set(split['train'])

def feed_extras(path):
    with open(path, encoding='utf-8') as f:
        feed = json.load(f)
    owner, out = feed['author'], []
    for p in feed['posts']:
        a = p.get('author')
        if isinstance(a, dict) and a.get('code') == owner:
            out.append((owner, p['urn'], p.get('commentsCount') or 0, p.get('repostsCount') or 0,
                        bool(p.get('resharedPost')), bool(p.get('article')), bool(p.get('isBrandPartnership'))))
    return out

with ProcessPoolExecutor() as ex:
    extras = pd.DataFrame(
        [r for rows in ex.map(feed_extras, sorted(glob.glob(f'{DATA_DIR}/posts/AUTHOR_*.json')), chunksize=64)
         for r in rows],
        columns=['author', 'post', 'comments', 'reposts', 'has_reshared', 'has_article', 'is_brand'])

feat = (features.build(author_level(posts), load_followers(DATA_DIR))
        .merge(extras, on=['author', 'post'], how='left', validate='one_to_one')
        .reset_index(drop=True))            # порядок: автор, время
for c in ('has_reshared', 'has_article', 'is_brand'):
    feat[c] = feat[c].astype(float)
feat['content_type'] = feat['content_type'].astype('category')

# Векторные истории через groupby без питоновских циклов.
g = feat.groupby('author', sort=False)
n_prev = g.cumcount().replace(0, np.nan)
for c in ('comments', 'reposts'):
    feat[f'log_{c}'] = np.log1p(feat[c])
    feat[f'hist_log_{c}'] = (g[f'log_{c}'].cumsum() - feat[f'log_{c}']) / n_prev
lags = np.column_stack([g['log_reactions'].shift(k).to_numpy() for k in (1, 2, 3)])
with warnings.catch_warnings():
    warnings.simplefilter('ignore', RuntimeWarning)
    feat['hist_last1'] = lags[:, 0] - feat['level']
    feat['hist_last3'] = np.nanmean(lags, axis=1) - feat['level']    # тренд: последние посты против медианы
feat['is_train_author'] = feat['author'].isin(TRAIN_AUTHORS)
print(feat.shape)

## 2. Эмбеддинги, выровненные по `feat`

Картинки поста усредняются одним умножением разреженной матрицы «пост × файл» на матрицу эмбеддингов.

In [ ]:
from scipy import sparse

def align(keys_df):
    # Позиция строки feat для каждой пары (author, post) из keys_df, -1 если нет.
    pos = feat[['author', 'post']].reset_index().rename(columns={'index': 'fi'})
    return keys_df.merge(pos, on=['author', 'post'], how='left')['fi'].fillna(-1).astype(int).to_numpy()

t = np.load(f'{WORK}/text_e5.npz', allow_pickle=True)
T = np.zeros((len(feat), t['emb'].shape[1]), np.float32)
fi = align(pd.DataFrame({'author': t['author'], 'post': t['post']}))
T[fi[fi >= 0]] = t['emb'][fi >= 0]
T_has = (feat['n_chars'] > 0).to_numpy() & (np.abs(T).sum(1) > 0)
MODS = {'text': (T, T_has)}

parts = sorted(glob.glob(f'{WORK}/img/part_*.npz'))
USE_IMAGES = bool(parts)
pm = pd.read_parquet(f'{WORK}/post_media.parquet')
pm['fi'] = align(pm)
pm = pm[pm['fi'] >= 0]
media_counts = (pd.crosstab(pm['fi'], pm['kind']).reindex(range(len(feat)), fill_value=0)
                .add_prefix('n_').reset_index(drop=True))
media_counts['n_reshared_media'] = (pm[pm['from_reshared_post']].groupby('fi').size()
                                    .reindex(range(len(feat)), fill_value=0).to_numpy())
if USE_IMAGES:
    loaded = [np.load(p, allow_pickle=True) for p in parts]
    sha = np.concatenate([x['sha'] for x in loaded])
    emb = np.concatenate([x['emb'] for x in loaded]).astype(np.float32)
    ok = np.concatenate([x['ok'] for x in loaded])
    ei = pd.Series(np.arange(len(sha)), index=sha)
    ei = ei[~ei.index.duplicated()]              # на случай повторно сохранённой части
    p_ok = pm[pm['sha'].isin(ei.index)]
    p_ok = p_ok.assign(ei=ei.loc[p_ok['sha']].to_numpy())
    p_ok = p_ok[ok[p_ok['ei']]]
    A = sparse.csr_matrix((np.ones(len(p_ok), np.float32), (p_ok['fi'], p_ok['ei'])), shape=(len(feat), len(sha)))
    I = np.asarray(A @ emb)
    norm = np.linalg.norm(I, axis=1, keepdims=True)
    I_has = norm[:, 0] > 0
    I[I_has] /= norm[I_has]
    MODS['img'] = (I, I_has)
    print(f'картинки: {len(sha):,} эмбеддингов, у {I_has.sum():,} постов есть хотя бы одна')
else:
    print('картинок пока нет, работаем только с текстом')

## 3. Фичи относительно истории автора

Для каждого автора одна матрица близостей `S = X·Xᵀ` по его постам, маска «только более ранние», и всё считается матрично:

* `sim_max`, `sim_prev`, `sim_centroid`: насколько пост похож на прошлые (на самый похожий, на предыдущий, на средний);
* `knn_soft`, `knn_top3`, `nn1`: как сыграли похожие прошлые посты, `log_reactions` минус текущий `level`;
* `hist_frac_above/below`: доля прошлых постов, которые по текущему уровню были бы above/below;
* `hist_same_type`: доля прошлых постов того же `content_type`.

Авторы раскидываются по процессам через joblib.

In [ ]:
from joblib import Parallel, delayed

TAU = 0.05
REL = ('sim_max', 'sim_prev', 'sim_centroid', 'knn_soft', 'knn_top3', 'nn1')

def author_block(lr, lvl, ct, mods):
    n = len(lr)
    L = np.tri(n, k=-1, dtype=bool)                    # L[i, j]: пост j раньше поста i
    cnt = np.arange(n, dtype=float)
    cnt[0] = np.nan
    R = lr[None, :] - lvl[:, None]                     # прошлые посты против текущего уровня
    out = {'hist_frac_above': ((R > HIGH) & L).sum(1) / cnt,
           'hist_frac_below': ((R < LOW) & L).sum(1) / cnt,
           'hist_same_type': ((ct[None, :] == ct[:, None]) & L).sum(1) / cnt}
    for name, (X, has) in mods.items():
        res = {k: np.full(n, np.nan) for k in REL}
        h = np.flatnonzero(has)
        if len(h) >= 2:
            Xh, m = X[h], len(h)
            S = Xh @ Xh.T
            Sm = np.where(np.tri(m, k=-1, dtype=bool), S, -np.inf)[1:]   # строки с историей
            rows, lr_h, lvl_r = h[1:], lr[h], lvl[h[1:]]
            res['sim_max'][rows] = Sm.max(1)
            res['sim_prev'][rows] = S[np.arange(1, m), np.arange(m - 1)]
            C = (np.cumsum(Xh, 0) - Xh)[1:]
            res['sim_centroid'][rows] = (Xh[1:] * C).sum(1) / np.linalg.norm(C, axis=1)
            W = np.exp((Sm - Sm.max(1, keepdims=True)) / TAU)            # exp(-inf) = 0
            res['knn_soft'][rows] = (W @ lr_h) / W.sum(1) - lvl_r
            k = min(3, m - 1)
            idx = np.argpartition(-Sm, k - 1, axis=1)[:, :k]
            top = np.take_along_axis(Sm, idx, 1)
            vals = np.where(np.isfinite(top), lr_h[idx], np.nan)
            res['knn_top3'][rows] = np.nanmean(vals, 1) - lvl_r
            res['nn1'][rows] = lr_h[idx[np.arange(len(rows)), top.argmax(1)]] - lvl_r
        out.update({f'{name}_{k}': v for k, v in res.items()})
    return out

lr, lvl = feat['log_reactions'].to_numpy(), feat['level'].to_numpy()
ct = feat['content_type'].cat.codes.to_numpy()
groups = list(feat.groupby('author', sort=False).indices.values())

def run(ix):
    return ix, author_block(lr[ix], lvl[ix], ct[ix], {k: (X[ix], has[ix]) for k, (X, has) in MODS.items()})

t0 = time.time()
blocks = Parallel(n_jobs=-1, batch_size=64)(delayed(run)(ix) for ix in groups)
rel = {}
for ix, out in blocks:
    for k, v in out.items():
        rel.setdefault(k, np.full(len(feat), np.nan))[ix] = v
rel = pd.DataFrame(rel)
print(f'{rel.shape[1]} фич за {time.time() - t0:.0f}s')

## 4. PCA эмбеддингов (обучается только на постах train-авторов)

In [ ]:
from sklearn.decomposition import PCA

N_PC = 32
train_rows = feat['is_train_author'].to_numpy()
pcs = []
for name, (X, has) in MODS.items():
    pca = PCA(N_PC, svd_solver='randomized', random_state=42).fit(X[has & train_rows])
    Z = np.full((len(feat), N_PC), np.nan, np.float32)
    Z[has] = pca.transform(X[has])
    pcs.append(pd.DataFrame(Z, columns=[f'{name}_pc{i}' for i in range(N_PC)]))
    print(f'{name}: PCA {N_PC} объясняет {pca.explained_variance_ratio_.sum():.0%}')

full = pd.concat([feat, rel, media_counts, *pcs], axis=1)
table = load_labels(DATA_DIR).merge(full, on=['author', 'post'], how='left', validate='one_to_one')
train = table[table['split'] == 'train'].reset_index(drop=True)
test = table[table['split'] == 'test'].reset_index(drop=True)
y, y_test = train['label'].to_numpy(), test['label'].to_numpy()
print(f'train {len(train):,}  test {len(test):,}')

## 5. Абляция на кросс-валидации по авторам

`GroupKFold` по `author` внутри train: так же, как train и test разделены по авторам. Наборы фич сравниваем
**по CV**, test трогаем один раз в конце. Фолды идут последовательно: `HistGradientBoosting` и так занимает все ядра через OpenMP,
параллельные фолды только мешали бы друг другу.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score

def make_model():
    return HistGradientBoostingClassifier(max_iter=600, learning_rate=0.05, max_leaf_nodes=31,
                                          l2_regularization=1.0, class_weight='balanced',
                                          categorical_features='from_dtype', random_state=42)

col = lambda prefix: [c for c in full.columns if c.startswith(prefix)]
BASE = features.CONTENT_NUM + features.HISTORY_NUM + features.ARTEFACT_NUM + features.CONTENT_CAT
META = ['has_reshared', 'has_article', 'is_brand', 'hist_log_comments', 'hist_log_reposts',
        'hist_last1', 'hist_last3', 'hist_frac_above', 'hist_frac_below', 'hist_same_type']
TEXT = col('text_')
IMG = col('img_') + list(media_counts.columns)
SETS = {'base': BASE, '+meta': BASE + META, '+meta+text': BASE + META + TEXT}
if USE_IMAGES:
    SETS['+meta+text+img'] = BASE + META + TEXT + IMG

folds = list(GroupKFold(n_splits=5).split(train, groups=train['author']))
def oof(cols):
    P = np.zeros((len(train), 3))
    for tr, va in folds:
        P[va] = make_model().fit(train.loc[tr, cols], y[tr]).predict_proba(train.loc[va, cols])
    return P

OOF = {}
for name, cols in SETS.items():
    t0 = time.time()
    OOF[name] = oof(cols)
    print(f'{name:16s} {len(cols):4d} фич  CV macro-F1 {f1_score(y, OOF[name].argmax(1), average="macro"):.4f}'
          f'  ({time.time() - t0:.0f}s)', flush=True)

## 6. Веса классов под macro-F1 и финальный прогноз

Сдвиг вероятностей `below` и `above` подбирается на OOF: сетка 41 × 41 считается векторно,
macro-F1 для всех вариантов сразу через `bincount` по матрицам ошибок.

In [ ]:
def macro_f1_many(y, preds):
    # preds: (G, n) предсказаний; macro-F1 для каждой из G строк без цикла по строкам.
    G = len(preds)
    flat = (np.arange(G)[:, None] * 9 + y[None, :] * 3 + preds).ravel()
    cm = np.bincount(flat, minlength=G * 9).reshape(G, 3, 3)
    tp = cm[:, [0, 1, 2], [0, 1, 2]]
    denom = cm.sum(2) + cm.sum(1)
    return (2 * tp / np.maximum(denom, 1)).mean(1)

def tune_weights(P, y, grid=np.exp(np.linspace(-0.7, 0.7, 41))):
    best = (-1.0, 1.0, 1.0)
    for w0 in grid:                                   # 41 векторных шагов по 41 варианту
        W = np.stack([np.full_like(grid, w0), np.ones_like(grid), grid], 1)     # (41, 3)
        preds = (P[None, :, :] * W[:, None, :]).argmax(2)                      # (41, n)
        f = macro_f1_many(y, preds)
        if f.max() > best[0]:
            best = (float(f.max()), float(w0), float(grid[f.argmax()]))
    return best

BEST = max(OOF, key=lambda k: f1_score(y, OOF[k].argmax(1), average='macro'))
f_cv, w0, w2 = tune_weights(OOF[BEST], y)
WEIGHTS = np.array([w0, 1.0, w2])
print(f'лучший набор {BEST}: CV macro-F1 после весов {f_cv:.4f}, веса below {w0:.2f}, above {w2:.2f}')

from kairos.metrics import score
results = {}
for name in ('base', BEST):
    cols = SETS[name]
    P = make_model().fit(train[cols], y).predict_proba(test[cols])
    w = WEIGHTS if name == BEST else np.ones(3)
    pred = (P * w).argmax(1)
    results[name] = score(y_test, pred)
    print(f'test {name:16s}', results[name])

os.makedirs(f'{REPO}/results/predictions', exist_ok=True)
out = test[['author', 'post']].assign(label=pred)
out.to_csv(f'{REPO}/results/predictions/model.csv', index=False)
out.to_csv(f'{WORK}/predictions_model.csv', index=False)

In [ ]:
%cd {REPO}
!python evaluate.py --data "$DATA_DIR" --predictions results/predictions/model.csv

## 7. Регрессия на `resid` и смесь с классификатором

Классы упорядочены и нарезаны из одного числа, `resid`. Регрессор предсказывает само отклонение от уровня автора,
а два порога подбираются на OOF под macro-F1. Потом пробуем смесь: `a · z(регрессия) + (1 − a) · z(P_above − P_below)`.
Вес `a` и пороги выбираются **по CV**, test считается один раз для выбранного варианта.

Подбор порогов векторный: для каждого нижнего порога все верхние считаются одним вызовом `macro_f1_many`.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import cohen_kappa_score

def make_reg():
    return HistGradientBoostingRegressor(max_iter=600, learning_rate=0.05, max_leaf_nodes=31,
                                         l2_regularization=1.0, categorical_features='from_dtype',
                                         random_state=42)

COLS = SETS[BEST]
r = train['resid'].to_numpy()
assert not np.isnan(r).any(), 'у размеченных постов resid должен быть всегда'

t0 = time.time()
R_OOF = np.zeros(len(train))
for tr, va in folds:
    R_OOF[va] = make_reg().fit(train.loc[tr, COLS], r[tr]).predict(train.loc[va, COLS])
print(f'регрессор OOF: R² {1 - ((r - R_OOF) ** 2).sum() / ((r - r.mean()) ** 2).sum():.4f}  ({time.time() - t0:.0f}s)')

def cut(s, lo, hi):
    return np.where(s < lo, 0, np.where(s > hi, 2, 1))

def tune_cuts(s, y, n=120):
    # Кандидаты в пороги: квантили s. На шаг цикла (нижний порог) все верхние сразу.
    q = np.unique(np.quantile(s, np.linspace(0.02, 0.98, n)))
    best = (-1.0, None, None)
    for i, lo in enumerate(q[:-1]):
        hi = q[i + 1:]
        preds = np.where(s[None, :] < lo, 0, np.where(s[None, :] > hi[:, None], 2, 1))
        f = macro_f1_many(y, preds)
        j = int(f.argmax())
        if f[j] > best[0]:
            best = (float(f[j]), float(lo), float(hi[j]))
    return best

zs = lambda s, ref: (s - ref.mean()) / ref.std()     # нормировка по OOF, та же применяется к test
S_CLF = OOF[BEST][:, 2] - OOF[BEST][:, 0]

print(f'{"":22s} {"CV macro-F1":>11s} {"CV QWK":>7s}')
print(f'{"классификатор + веса":22s} {f_cv:11.4f} {cohen_kappa_score(y, (OOF[BEST] * WEIGHTS).argmax(1), weights="quadratic"):7.4f}')
BLEND = {}
for a in (0.0, 0.25, 0.5, 0.75, 1.0):
    s = a * zs(R_OOF, R_OOF) + (1 - a) * zs(S_CLF, S_CLF)
    f, lo, hi = tune_cuts(s, y)
    BLEND[a] = (f, lo, hi)
    print(f'{f"смесь a={a:.2f}":22s} {f:11.4f} {cohen_kappa_score(y, cut(s, lo, hi), weights="quadratic"):7.4f}')

In [ ]:
A = max(BLEND, key=lambda a: BLEND[a][0])
f_blend, lo, hi = BLEND[A]
if f_blend <= f_cv:
    print(f'смесь по CV не лучше классификатора с весами ({f_blend:.4f} против {f_cv:.4f}), test не считаем')
else:
    print(f'выбрано a={A:.2f}, CV macro-F1 {f_blend:.4f} (было {f_cv:.4f})')
    P_test = make_model().fit(train[COLS], y).predict_proba(test[COLS])
    R_test = make_reg().fit(train[COLS], r).predict(test[COLS])
    s_test = A * zs(R_test, R_OOF) + (1 - A) * zs(P_test[:, 2] - P_test[:, 0], S_CLF)
    pred_reg = cut(s_test, lo, hi)
    print('test', score(y_test, pred_reg))

    rng = np.random.default_rng(0)
    codes, uniq = pd.factorize(test['author'])
    by_author = np.split(np.argsort(codes, kind='stable'), np.cumsum(np.bincount(codes))[:-1])
    samples = [np.concatenate([by_author[i] for i in rng.integers(0, len(uniq), len(uniq))]) for _ in range(1000)]
    for name, p in (('классификатор', pred), ('смесь', pred_reg)):
        boot = [f1_score(y_test[s], p[s], average='macro') for s in samples]
        print(f'  {name:14s} test macro-F1 95%: {np.percentile(boot, [2.5, 97.5]).round(4)}')
    # Те же выборки авторов для обеих моделей: интервал разницы честнее, чем сравнение двух интервалов.
    diff = [f1_score(y_test[s], pred_reg[s], average='macro') - f1_score(y_test[s], pred[s], average='macro')
            for s in samples]
    print(f'  разница смесь − классификатор, 95%: {np.percentile(diff, [2.5, 97.5]).round(4)}')

    out = test[['author', 'post']].assign(label=pred_reg)
    out.to_csv(f'{REPO}/results/predictions/model_blend.csv', index=False)
    out.to_csv(f'{WORK}/predictions_model_blend.csv', index=False)

In [ ]:
%cd {REPO}
!test -f results/predictions/model_blend.csv && python evaluate.py --data "$DATA_DIR" --predictions results/predictions/model_blend.csv